### Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, RNN,GRU
import tensorflow as tf

### Custom GRU definition

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer

class CustomGRUCell(Layer):
    def __init__(self, units, **kwargs):
        super(CustomGRUCell, self).__init__(**kwargs)
        self.units = units
        self.state_size = units  # Set the state_size attribute

    def build(self, input_shape):
        input_dim = input_shape[-1]
        # Defining trainable parameters
        self.Wr = self.add_weight(shape=(input_dim, self.units), initializer='random_normal', name='Wr')
        self.Ur = self.add_weight(shape=(self.units, self.units), initializer='random_normal', name='Ur')
        self.br = self.add_weight(shape=(self.units,), initializer='zeros', name='br')

        self.Wz = self.add_weight(shape=(input_dim, self.units), initializer='random_normal', name='Wz')
        self.Uz = self.add_weight(shape=(self.units, self.units), initializer='random_normal', name='Uz')
        self.bz = self.add_weight(shape=(self.units,), initializer='zeros', name='bz')

        self.Wh = self.add_weight(shape=(input_dim, self.units), initializer='random_normal', name='Wh')
        self.Uh = self.add_weight(shape=(self.units, self.units), initializer='random_normal', name='Uh')
        self.bh = self.add_weight(shape=(self.units,), initializer='zeros', name='bh')

        # Including trainable parameters for the deepfake gate
        self.Wd = self.add_weight(shape=(input_dim, self.units), initializer='random_normal', name='Wd')
        self.Ud = self.add_weight(shape=(self.units, self.units), initializer='random_normal', name='Ud')
        self.bd = self.add_weight(shape=(self.units,), initializer='zeros', name='bd')

        super(CustomGRUCell, self).build(input_shape)

    def call(self, inputs, states):
        h_prev = states[0]
        input_dim = inputs.shape[-1]
        inputs_df = tf.concat([tf.zeros_like(inputs[:, :input_dim-1]), inputs[:, -1:]], axis=-1)
        
        # Reset gate
        r = tf.sigmoid(tf.matmul(inputs, self.Wr) + tf.matmul(h_prev, self.Ur) + self.br)

        # Update gate
        z = tf.sigmoid(tf.matmul(inputs, self.Wz) + tf.matmul(h_prev, self.Uz) + self.bz)

        # deepfake gate
        d = tf.sigmoid(tf.matmul(inputs_df, self.Wd) + tf.matmul(h_prev, self.Ud) + self.bd)

        # Candidate hidden state
        h_tilde = tf.tanh(tf.matmul(inputs, self.Wh) + tf.matmul(r * h_prev, self.Uh) +  tf.matmul(d * h_prev, self.Uh) + self.bh)
        
        # Updated hidden state
        h = (1 - z) * h_prev + z * h_tilde 
        return h, [h]

    def get_config(self):
        config = super(CustomGRUCell, self).get_config()
        config.update({'units': self.units})
        return config


### Loading the data

In [ ]:
data = pd.read_csv("") # load the features used by the model here
class_pred = pd.read_csv("")# load CNN predictions here 
data['class_pred'] = class_pred # concatante class preds to the main dataset
data.head()

### Windowing the data for the usecase


In [ ]:
def create_windows(data,labels, window_size):
    sequences = []
    classes = []
    for i in range(0,len(data) - window_size + 1,window_size):
        sequence = data.iloc[i:i + window_size]
        sequences.append(sequence)
        classes.append(labels[i+window_size-1])
    return np.array(sequences), np.array(classes)

# Set the window size
window_size = 5

# Apply the function to create the times_series data
features = data.drop('class', axis=1)
time_series,classes = create_windows(features, labels, window_size)
print("Time Series Data Shape:", time_series.shape)
print("Labels Shape:", classes.shape)

### Split the daa

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(time_series, classes, test_size=0.1, random_state=42)
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, test_size=0.2, random_state=42)

### Building the model

In [ ]:
prec = tf.keras.metrics.Precision()
rec = tf.keras.metrics.Recall()

In [ ]:
def build_and_train(X_train, Y_train, X_val, Y_val, epochs, optimizer = 'adam'):
    model = Sequential()
    model.add(RNN(CustomGRUCell(units=256), return_sequences=True, input_shape=(5, 350)))
    model.add(GRU(128))
    model.add(Dense(units=64, activation='relu'))
    model.add(Dense(units=1, activation='sigmoid'))
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy', prec,rec])
    model.summary()
    hist = model.fit(X_train, Y_train, epochs=epochs, validation_data=(X_val, Y_val))
    return model, hist

### Training the model with default Adam optimizer

In [ ]:
model,hist =build_and_train(X_train, Y_train, X_val, Y_val, 10)

### Training the model with Another optimizer

In [ ]:
model_sgd,hist_sgd =build_and_train(X_train, Y_train, X_val, Y_val, 200, 'SGD')